In [13]:
pip install pandas requests

Note: you may need to restart the kernel to use updated packages.


In [19]:
import pandas as pd
import requests
import time
import collections
from urllib.parse import urlparse

# ---------------------------------------------------------
# PLATFORM SPECIFIC SCRAPING LOGIC
# ---------------------------------------------------------

def fetch_greenhouse_jobs(company_name, api_url):
    jobs_list = []
    try:
        response = requests.get(api_url, timeout=10)
        if response.status_code == 200:
            jobs = response.json().get('jobs', [])
            for job in jobs:
                jobs_list.append({
                    'Company': company_name, 'Job Title': job.get('title'),
                    'Location': job.get('location', {}).get('name', 'Unknown'),
                    'Job URL': job.get('absolute_url'), 'Updated At': job.get('updated_at')
                })
    except Exception as e: print(f"[{company_name}] Error: {e}")
    return jobs_list

def fetch_lever_jobs(company_name, api_url):
    jobs_list = []
    try:
        response = requests.get(api_url, timeout=10)
        if response.status_code == 200:
            for job in response.json():
                jobs_list.append({
                    'Company': company_name, 'Job Title': job.get('text'),
                    'Location': job.get('categories', {}).get('location', 'Unknown'),
                    'Job URL': job.get('hostedUrl'), 'Updated At': job.get('createdAt')
                })
    except Exception as e: print(f"[{company_name}] Error: {e}")
    return jobs_list

def fetch_ashby_jobs(company_name, api_url):
    jobs_list = []
    try:
        response = requests.get(api_url, timeout=10)
        if response.status_code == 200:
            for job in response.json().get('jobs', []):
                jobs_list.append({
                    'Company': company_name, 'Job Title': job.get('title'),
                    'Location': job.get('location', 'Unknown'),
                    'Job URL': job.get('jobUrl'), 'Updated At': job.get('publishedAt')
                })
    except Exception as e: print(f"[{company_name}] Error: {e}")
    return jobs_list

def fetch_teamtailor_jobs(company_name, api_url):
    jobs_list = []
    try:
        headers = {'Accept': 'application/vnd.api+json', 'X-Api-Version': '20210218'}
        response = requests.get(api_url, headers=headers, timeout=10)
        if response.status_code == 200:
            for job in response.json().get('data', []):
                attr = job.get('attributes', {})
                jobs_list.append({
                    'Company': company_name, 'Job Title': attr.get('title'),
                    'Location': attr.get('remote-status', 'Unknown'),
                    'Job URL': job.get('links', {}).get('careersite-job-url'),
                    'Updated At': attr.get('created-at')
                })
    except Exception as e: print(f"[{company_name}] Error: {e}")
    return jobs_list

def fetch_eightfold_jobs(company_name, api_url):
    jobs_list = []
    offset = 0
    try:
        while True:
            url = f"{api_url}&start={offset}" if "?" in api_url else f"{api_url}?start={offset}"
            data = requests.get(url, timeout=10).json()
            positions = data.get('positions', [])
            if not positions: break
            for pos in positions:
                jobs_list.append({
                    'Company': company_name, 'Job Title': pos.get('name'),
                    'Location': pos.get('location'), 'Job URL': pos.get('canonicalPositionUrl')
                })
            offset += 10
            time.sleep(0.5)
    except Exception as e: print(f"[{company_name}] Error: {e}")
    return jobs_list

def fetch_gatsby_i18n_jobs(company_name, api_url):
    jobs_list = []
    try:
        data = requests.get(api_url, timeout=10).json()
        extracted = collections.defaultdict(dict)
        def find_keys(node):
            if isinstance(node, dict):
                for k, v in node.items():
                    if k.startswith('career.job.jobsInfo.'):
                        parts = k.split('.')
                        if len(parts) >= 5: extracted[parts[3]][parts[4]] = v
                    else: find_keys(v)
            elif isinstance(node, list):
                for item in node: find_keys(item)
        find_keys(data)
        for slug, job in extracted.items():
            if 'hong kong' in str(job.get('location', '')).lower():
                jobs_list.append({
                    'Company': company_name, 'Job Title': job.get('title'),
                    'Location': job.get('location'), 'Department': job.get('department'),
                    'Job URL': f"https://www.aqumon.com/about/career/{slug}"
                })
    except Exception as e: print(f"[{company_name}] Error: {e}")
    return jobs_list

def fetch_oracle_hcm_jobs(company_name, api_url):
    jobs_list = []
    offset, limit = 0, 25
    try:
        while True:
            params = {"onlyData": "true", "expand": "requisitionList.workLocation",
                      "finder": "findReqs;siteNumber=CX_3002,sortBy=POSTING_DATES_DESC",
                      "limit": limit, "offset": offset}
            data = requests.get(api_url, params=params, timeout=10).json()
            if 'items' not in data or not data['items']: break
            root = data['items'][0]
            batch = root.get('requisitionList', [])
            if not batch: break
            for job in batch:
                jid = job.get('Id')
                jobs_list.append({
                    'Company': company_name, 'Job Title': job.get('Title'),
                    'Location': job.get('PrimaryLocation'), 'Updated At': job.get('PostedDate'),
                    'Job URL': f"https://hdpc.fa.us2.oraclecloud.com/hcmUI/CandidateExperience/en/sites/CX_3002/job/{jid}"
                })
            if offset + len(batch) >= root.get('TotalJobsCount', 0): break
            offset += limit
            time.sleep(1)
    except Exception as e: print(f"[{company_name}] Error: {e}")
    return jobs_list

def fetch_hkex_workday_jobs(company_name):
    jobs_list = []
    base, api = "https://hkex.wd3.myworkdayjobs.com/HKEXCareerPage", "https://hkex.wd3.myworkdayjobs.com/wday/cxs/hkex/HKEXCareerPage/jobs"
    s = requests.Session()
    try:
        s.get(base, timeout=10)
        csrf = s.cookies.get("CALYPSO_CSRF_TOKEN")
        offset, total = 0, None
        while True:
            payload = {"appliedFacets": {}, "limit": 20, "offset": offset, "searchText": ""}
            data = s.post(api, json=payload, headers={"x-calypso-csrf-token": csrf}, timeout=10).json()
            if total is None: total = data.get("total", 0)
            batch = data.get("jobPostings", [])
            if not batch: break
            for j in batch:
                jobs_list.append({
                    'Company': company_name, 'Job Title': j.get('title'),
                    'Location': j.get('locationsText'), 'Updated At': j.get('postedOn'),
                    'Job URL': f"{base}{j.get('externalPath')}"
                })
            if len(jobs_list) >= total: break
            offset += 20
            time.sleep(1)
    except Exception as e: print(f"[{company_name}] Workday Error: {e}")
    return jobs_list

# ---------------------------------------------------------
# MAIN
# ---------------------------------------------------------

def main():
    df = pd.read_csv("target_companies_api.csv")
    valid_apis = df[df['API Endpoint'].notna() & (df['API Endpoint'] != 'N/A') & (df['API Endpoint'] != '')]
    all_results = []

    for _, row in valid_apis.iterrows():
        comp, plat, url = row['Company'], str(row['Platform']).lower(), row['API Endpoint']
        print(f"Scraping {comp}...")
        
        # Mapping Company/Platform to correct Function
        if comp == "HKEX": jobs = fetch_hkex_workday_jobs(comp)
        elif 'greenhouse' in plat: jobs = fetch_greenhouse_jobs(comp, url)
        elif 'lever' in plat: jobs = fetch_lever_jobs(comp, url)
        elif 'ashby' in plat: jobs = fetch_ashby_jobs(comp, url)
        elif 'teamtailor' in plat: jobs = fetch_teamtailor_jobs(comp, url)
        elif 'eightfold' in plat: jobs = fetch_eightfold_jobs(comp, url)
        elif 'gatsby' in plat: jobs = fetch_gatsby_i18n_jobs(comp, url)
        elif 'oracle' in plat: jobs = fetch_oracle_hcm_jobs(comp, url)
        else:
            print(f"-> Skipping {comp}: Unknown Platform logic.")
            continue
            
        print(f"-> Found {len(jobs)} jobs.")
        all_results.extend(jobs)

    pd.DataFrame(all_results).to_csv("master_jobs_list.csv", index=False)
    print(f"\nDone! Scraped {len(all_results)} jobs total.")

if __name__ == "__main__":
    main()

Scraping Crypto.com...
-> Found 43 jobs.
Scraping OKX...
-> Found 305 jobs.
Scraping Airwallex...
[Airwallex] Error: HTTPSConnectionPool(host='api.ashbyhq.com', port=443): Read timed out. (read timeout=10)
-> Found 0 jobs.
Scraping Reap...
-> Found 10 jobs.
Scraping AQUMON...
-> Found 20 jobs.
Scraping Goldman Sachs...
-> Found 1350 jobs.
Scraping HSBC...
-> Found 1803 jobs.
Scraping Point72...
-> Found 270 jobs.
Scraping HKMA...
-> Skipping HKMA: Unknown Platform logic.
Scraping HKEX...
-> Found 177 jobs.
Scraping Chainalysis...
-> Found 43 jobs.

Done! Scraped 4021 jobs total.


In [ ]:
jobs_csv = "master_jobs_list.csv"
try:
    df_jobs = pd.read_csv(jobs_csv)
except FileNotFoundError:
    print(f"Error: Could not find {jobs_csv}.")
    
filtered_df = df_jobs[df_jobs['Location'].str.lower().str.contains('hong', na=False) | df_jobs['Company'].str.lower().str.contains('hk', na=False)]
output_file = "master_jobs_list_filtered.csv"
filtered_df.to_csv(output_file, index=False)